Modelo predictivo manual

In [8]:
import torch
import torch.nn as nn
import joblib
import numpy as np

# Arquitectura del entrenamiento
class MLPPaper(nn.Module):
    def __init__(self, in_dim=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

def predecir_manual(Ue, dcpdx, delta, tau_w, Tu_BL, freq):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Ruta del los archivos del modelo
    ruta_scaler = 'scaler.pkl'
    ruta_modelo = 'modelo_mlp.pth'

    # Cargar
    scaler = joblib.load(ruta_scaler)
    model = MLPPaper(in_dim=6).to(device)
    model.load_state_dict(torch.load(ruta_modelo, map_location=device))
    model.eval()

    X_manual = np.array([[Ue, dcpdx, delta, tau_w, Tu_BL, freq]])
    X_scaled = scaler.transform(X_manual).astype(np.float32)
    X_tensor = torch.tensor(X_scaled).to(device)

    with torch.no_grad():
        prediccion = model(X_tensor).cpu().numpy().ravel()[0]

    return prediccion

# Insertar valores de las variables manualmente

Ue     = 15.5      # Velocidad Ue
dcpdx  = 0.001     # Gradiente dcpdx
delta  = 0.012     # Espesor delta
tau_w  = 0.85      # tau_w
Tu_BL  = 0.03      # Turbulencia Tu_BL
freq   = 1200.0    # Frecuencia (Hz)

# Resultado de la predicción
resultado = predecir_manual(Ue, dcpdx, delta, tau_w, Tu_BL, freq)
print(f"La predicción (spec) para estas variables es: {resultado:.4f}")

La predicción (spec) para estas variables es: ----


Modelo predictivo con dataset

In [10]:
import torch
import torch.nn as nn
import joblib
import numpy as np
import pandas as pd
import scipy.io as sio
import h5py
from google.colab import files
from IPython.display import display

# Arquitectura del entrenamiento
class MLPPaper(nn.Module):
    def __init__(self, in_dim=6):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

def ejecutar_prediccion_dataset():

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ruta_scaler = 'scaler.pkl'
    ruta_modelo = 'modelo_mlp.pth'

    try:
        scaler = joblib.load(ruta_scaler)
        model = MLPPaper(in_dim=6).to(device)
        model.load_state_dict(torch.load(ruta_modelo, map_location=device))
        model.eval()
    except FileNotFoundError:
        print("¡Error!")
        return

    # 3. Subir el archivo .mat correspondiente al dataset
    print("Selecciona y sube tu archivo .mat:")
    uploaded = files.upload()

    if not uploaded:
        print("No se subió ningún archivo.")
        return

    ruta_mat = list(uploaded.keys())[0]

    try:
        mat_data = sio.loadmat(ruta_mat)
        data_mat = mat_data['matriz_original']
    except NotImplementedError:
        with h5py.File(ruta_mat, "r") as f:
            data_mat = np.array(f['matriz_original']).T

    # Preparar el DataFrame
    features = ["Ue", "dcpdx", "delta", "tau_w", "Tu_BL", "freq"]
    # Se asume que la matriz con 7columnas, ajustar para solo 6.
    df_naca = pd.DataFrame(data_mat, columns=features + ["spec_real"])
    df_naca = df_naca.dropna().reset_index(drop=True)

    X_datos = df_naca[features].values
    X_scaled = scaler.transform(X_datos).astype(np.float32)
    X_tensor = torch.tensor(X_scaled).to(device)

    with torch.no_grad():
        preds = model(X_tensor).cpu().numpy().ravel()

    df_naca['Prediccion_Spec'] = preds

    print("PREDICCIÓN")

    # Generación de tabla
    respuesta = input("Tabla con predicciones?")

    if respuesta.strip().lower() == 's':
        display(df_naca)

    return df_naca

# Ejecutar la celda
df_resultados_mat = ejecutar_prediccion_dataset()

Por favor, selecciona y sube tu archivo .mat:


Saving Dataset_NACA0012_____v3.mat to Dataset_NACA0012_____v3.mat

¡Predicción finalizada con éxito!
¿Deseas visualizar la tabla con las variables y predicciones de cada fila? (s/n): s


,Ue,dcpdx,delta,tau_w,Tu_BL,freq,spec_real,Prediccion_Spec
0,9.7142,1.147353,0.006005,0.201635,10.0,112.139120,7.820495,1.713489
1,9.7142,1.147353,0.006005,0.201635,10.0,120.111141,7.576104,1.473508
2,9.7142,1.147353,0.006005,0.201635,10.0,128.649896,7.331714,1.303839
3,9.7142,1.147353,0.006005,0.201635,10.0,137.795675,7.372446,1.151571
4,9.7142,1.147353,0.006005,0.201635,10.0,147.591632,7.698300,0.988484
...,...,...,...,...,...,...,...,...
204,9.7142,1.147353,0.006005,0.201635,20.0,8154.660574,6.986387,10.056787
205,9.7142,1.147353,0.006005,0.201635,20.0,8487.982495,7.380662,10.512716
206,9.7142,1.147353,0.006005,0.201635,20.0,8885.637138,7.707036,11.396053
207,9.7142,1.147353,0.006005,0.201635,20.0,9248.837742,7.993217,12.333448
